# Notebook 018

# Extending Scientific Object Profiles with the NASA/IPAC Extragalactic Database (NED)

## Research Question

What additional scientific information can the NASA/IPAC Extragalactic Database (NED) provide about Einstein's first identified galaxy?

---

## Background

In Notebook 017, Einstein successfully identified its first astronomical object using the SIMBAD database. While SIMBAD specializes in object identification and literature references, NED focuses specifically on galaxies and other extragalactic objects.

NED integrates measurements from numerous astronomical surveys and publications, providing information such as redshift, alternative catalog identifiers, object classifications, photometry, and links to scientific literature.

By combining Einstein's own measurements with information from NED, the Scientific Object Profile can evolve from a simple identification record into a richer scientific dossier.

---

## Investigation Plan

This notebook will:

1. Load Einstein's Scientific Object Profile.
2. Query NED using the object's celestial coordinates.
3. Identify the corresponding extragalactic object.
4. Retrieve additional scientific information.
5. Extend the Scientific Object Profile.
6. Save the enhanced profile for future investigations.

---

## Expected Outcome

By the end of this notebook, Einstein will have combined its own image-based measurements with information from both SIMBAD and NED, creating a more comprehensive scientific profile of its first investigated galaxy.

In [1]:
# ============================================================
# Cell 2 - Import Libraries
# ============================================================

from astroquery.ipac.ned import Ned

from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

import pandas as pd

In [2]:
# ============================================================
# Cell 3 - Load Einstein's Scientific Object Profile
# ============================================================

profile = Table.read("../data/catalogs/object45_profile.ecsv")

print("Scientific profile loaded.\n")
print(profile)

Scientific profile loaded.

Einstein Label   SIMBAD Name   Object Type ... SIMBAD RA SIMBAD Dec
-------------- --------------- ----------- ... --------- ----------
            45 [RTG2015] 24643           G ... 53.171235 -27.771108


In [3]:
# ============================================================
# Cell 4 - Recover Einstein Coordinates
# ============================================================

coord = SkyCoord(
    ra=float(profile["Einstein RA (deg)"][0]) * u.deg,
    dec=float(profile["Einstein Dec (deg)"][0]) * u.deg,
)

print("Einstein Coordinates")
print("---------------------")
print(coord)

Einstein Coordinates
---------------------
<SkyCoord (ICRS): (ra, dec) in deg
    (53.16993599, -27.77112719)>


In [4]:
# ============================================================
# Cell 5 - Query NED
# ============================================================

result = Ned.query_region(
    coord,
    radius=5 * u.arcsec
)

print("Number of objects returned:", len(result))
result

Number of objects returned: 52


No.,Object Name,RA,DEC,Type,Velocity,Redshift,Redshift Flag,Magnitude and Filter,Separation,References,Notes,Photometry Points,Positions,Redshift Points,Diameter Points,Associations
,,degrees,degrees,,km / s,,,,arcmin,,,,,,,
int32,str30,float64,float64,object,float64,float64,object,object,float64,int32,int32,int32,int32,int32,int32,int32
1,UDF:[JBM2015] 31340,53.16852,-27.77103,VisS,329982.0,1.1007,PUN,,0.076,1,0,4,1,1,0,0
2,UDF:[JBM2015] 31312,53.1686,-27.77115,VisS,367995.0,1.2275,PUN,,0.071,1,0,3,1,1,0,0
3,UDF:[JBM2015] 31515,53.16862,-27.77069,VisS,356153.0,1.188,PUN,,0.075,1,0,5,1,1,0,0
4,CANDELS J033240.47-274615.0,53.16865,-27.77085,VisS,351357.0,1.172,PUN,29.3V,0.071,6,0,13,4,3,0,0
5,UDF:[JBM2015] 31304,53.1687,-27.77098,VisS,356153.0,1.188,PUN,,0.066,1,0,5,1,1,0,0
6,UDF:[JBM2015] 31346,53.16875,-27.77092,VisS,357473.0,1.1924,PUN,,0.065,1,0,3,1,1,0,0
7,UDF:[JBM2015] 31458,53.16876,-27.7706,VisS,353246.0,1.1783,PUN,,0.07,1,0,2,1,1,0,0
8,UDF:[CBS2006] 07837,53.16879,-27.77162,VisS,302790.0,1.01,PUN,29.2V,0.068,3,0,13,2,2,0,0


In [10]:
# ============================================================
# Cell 6 - Find the Closest NED Match
# ============================================================

import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

# Build SkyCoord from the NED table
coords = SkyCoord(
    ra=result["RA"].data,
    dec=result["DEC"].data,
    unit="deg"
)

# Calculate separation from Einstein's coordinates
separation = coord.separation(coords)

# Add the separation to the table
result["Einstein Separation (arcsec)"] = separation.arcsec

# Find the closest match
closest = result[np.argmin(separation)]

print("Closest NED Match")
print("-----------------")
print(closest)

Closest NED Match
-----------------
No.         Object Name              RA        DEC     Type Velocity Redshift Redshift Flag Magnitude and Filter Separation References Notes Photometry Points Positions Redshift Points Diameter Points Associations Einstein Separation (arcsec)
                                  degrees    degrees         km / s                                                arcmin                                                                                                                         
--- ---------------------------- ---------- ---------- ---- -------- -------- ------------- -------------------- ---------- ---------- ----- ----------------- --------- --------------- --------------- ------------ ----------------------------
 29 GALEXMSC J033240.76-274616.1   53.16992  -27.77112  UvS       --       --                                         0.002          0     0                14         2               0               0            0          0.0571236377

In [11]:
# ============================================================
# Cell 7 - Five Closest NED Objects
# ============================================================

sorted_result = result.copy()
sorted_result.sort("Einstein Separation (arcsec)")

sorted_result[:5]

No.,Object Name,RA,DEC,Type,Velocity,Redshift,Redshift Flag,Magnitude and Filter,Separation,References,Notes,Photometry Points,Positions,Redshift Points,Diameter Points,Associations,Einstein Separation (arcsec)
,,degrees,degrees,,km / s,,,,arcmin,,,,,,,,
int32,str30,float64,float64,object,float64,float64,object,object,float64,int32,int32,int32,int32,int32,int32,int32,float64
29,GALEXMSC J033240.76-274616.1,53.16992,-27.77112,UvS,--,--,,,0.002,0,0,14,2,0,0,0,0.05712363776300755
32,WISEA J033240.79-274616.2,53.16999,-27.77117,IrS,--,--,,,0.004,0,0,12,1,0,0,0,0.23098349785672098
30,EIS J033240.79-274615.9,53.16992,-27.77103,G,186475.0,0.622013,SLS,22.1R,0.006,54,0,126,28,35,0,0,0.35356920493350597
28,SSTSL2 J033240.77-274616.4,53.16991,-27.77124,IrS,--,--,,,0.007,0,0,10,1,0,0,0,0.4144672979828138
36,ASPECS-LP.1 mm.C28,53.17021,-27.77122,SmmS,186471.0,0.622,SUN,,0.015,2,0,0,2,1,0,0,0.9345887359342863


# Multi-Wavelength View of the Galaxy

One of the most important discoveries in this investigation is that a single astronomical object may appear in many different catalogs, each representing observations made at different wavelengths of light.

Although Einstein began with a single object detected in a Hubble Space Telescope image, a search of the NASA/IPAC Extragalactic Database (NED) revealed several nearby catalog entries representing observations of the same region of the sky.

These include observations from multiple astronomical missions:

| Survey | Wavelength | Scientific Information |
|--------|------------|------------------------|
| **GALEX** | Ultraviolet | Young, hot stars and recent star formation |
| **WISE** | Infrared | Warm dust and stellar populations |
| **Spitzer (SSTSL2)** | Infrared | Dust, star-forming regions, and embedded objects |
| **EIS** | Optical | Galaxy morphology, photometry, and redshift |
| **ASPECS / ALMA** | Millimeter | Cold molecular gas and the raw material for future star formation |

This demonstrates an important principle of modern astronomy:

> **Astronomical objects are not fully understood by observing them in only one wavelength of light.**

Different wavelengths reveal different physical processes occurring within the same galaxy.

For example:

- Ultraviolet observations trace regions of active star formation.
- Optical observations reveal the overall structure and morphology of the galaxy.
- Infrared observations penetrate dust and reveal cooler stellar populations.
- Millimeter observations detect cold gas from which new stars may eventually form.

Einstein has therefore moved beyond simply identifying an astronomical object. By combining information from multiple astronomical surveys, it has begun constructing a **multi-wavelength scientific profile** that provides a richer understanding of the galaxy than any single catalog can offer.

This notebook marks an important milestone in the development of Einstein as an AI-assisted astronomical research system. Future notebooks will continue to integrate additional databases and observations, allowing Einstein to build increasingly comprehensive scientific dossiers for galaxies throughout the Universe.

In [12]:
# ============================================================
# Cell 8 - Selected NED Galaxy Information
# ============================================================

galaxy = result[result["Object Name"] == "EIS J033240.79-274615.9"][0]

print("=" * 60)
print("Selected NED Galaxy")
print("=" * 60)

print(f"Object Name : {galaxy['Object Name']}")
print(f"Type        : {galaxy['Type']}")
print(f"Redshift    : {galaxy['Redshift']}")
print(f"Velocity    : {galaxy['Velocity']} km/s")
print(f"References  : {galaxy['References']}")
print(f"Photometry  : {galaxy['Photometry Points']}")

Selected NED Galaxy
Object Name : EIS J033240.79-274615.9
Type        : G
Redshift    : 0.622013
Velocity    : 186475.0 km/s
References  : 54
Photometry  : 126


In [13]:
# ============================================================
# Cell 9 - Coordinate Comparison
# ============================================================

from astropy.coordinates import SkyCoord
import astropy.units as u

einstein = SkyCoord(
    profile["Einstein RA (deg)"][0],
    profile["Einstein Dec (deg)"][0],
    unit="deg"
)

simbad = SkyCoord(
    float(profile["SIMBAD RA"][0]),
    float(profile["SIMBAD Dec"][0]),
    unit="deg"
)

ned = SkyCoord(
    galaxy["RA"],
    galaxy["DEC"],
    unit="deg"
)

print("=" * 60)
print("Coordinate Comparison")
print("=" * 60)

print(f"Einstein → SIMBAD : {einstein.separation(simbad).arcsec:.3f} arcsec")
print(f"Einstein → NED    : {einstein.separation(ned).arcsec:.3f} arcsec")
print(f"SIMBAD → NED      : {simbad.separation(ned).arcsec:.3f} arcsec")

Coordinate Comparison
Einstein → SIMBAD : 4.138 arcsec
Einstein → NED    : 0.354 arcsec
SIMBAD → NED      : 4.198 arcsec


# Conclusions

Notebook 018 expanded Einstein's first Scientific Object Profile by incorporating information from the NASA/IPAC Extragalactic Database (NED).

Rather than identifying a single catalog entry, Einstein discovered that the same galaxy has been observed by multiple astronomical surveys operating at different wavelengths. These observations include ultraviolet, optical, infrared, and millimeter data, each revealing different physical properties of the galaxy.

This investigation demonstrates that modern astronomical research depends on integrating information from many complementary surveys rather than relying on a single observation.

Einstein now possesses its first **multi-wavelength scientific profile**, combining:

- Image measurements derived directly from Hubble observations.
- Identification through SIMBAD.
- Extragalactic information from NED.
- Evidence from multiple astronomical surveys.

This notebook represents another important step toward an AI-assisted astronomical research workflow capable of synthesizing information across the astronomical literature and archival databases.

In [14]:
# ============================================================
# Notebook Status
# ============================================================

from datetime import datetime

print("=" * 60)
print("Notebook Status")
print("=" * 60)

print("Status    : PASS")
print("Notebook  : 18_NED_scientific_profiles.ipynb")
print("Completed :", datetime.now().strftime("%Y-%m-%d %H:%M"))

Notebook Status
Status    : PASS
Notebook  : 18_NED_scientific_profiles.ipynb
Completed : 2026-07-19 12:14
